In [0]:
spark.conf.set(
    "***"
)

In [0]:
silver_path = "abfss://processed-data@adarshpmstorage01.dfs.core.windows.net/silver"

gold_df = spark.read.format("delta").load(silver_path)

display(gold_df)

timestamp,machine_id,assembly_line_no,hydraulic_pressure_bar,coolant_pressure_bar,air_system_pressure_bar,coolant_temperature,hydraulic_oil_temperature_c,spindle_bearing_temperature_c,spindle_vibration_m,tool_vibration_m,spindle_speed_rpm,voltage_volts,torque_nm,cutting_kn,downtime
11-01-2022 02:41:45,Makino-Synth-00044,Shopfloor-L2,116.177,3.8079,5.7951,20.7298,49.5432,38.0954,1.395,31.4333,16984.0576,311.8253,24.9367,3.1128,No_Machine_Failure
11-04-2023 16:19:39,Makino-Synth-00725,Shopfloor-L4,173.0649,4.396,6.7532,29.7766,50.7092,38.96,1.6492,36.2306,15562.4911,302.6555,27.2346,3.0019,No_Machine_Failure
14-06-2024 02:43:22,Makino-Synth-00199,Shopfloor-L8,122.3777,5.8749,6.6064,30.8863,55.3732,38.3555,1.3045,32.2337,20422.081,345.2439,19.4845,2.8946,No_Machine_Failure
25-09-2022 17:58:49,Makino-Synth-00102,Shopfloor-L8,97.7814,6.1863,6.6692,0.4788,49.2437,27.1742,1.5554,50.3614,15432.4345,343.0976,38.2867,3.7489,No_Machine_Failure
26-01-2022 00:36:35,Makino-Synth-00745,Shopfloor-L7,213.219,5.1724,6.1611,14.7283,50.0174,39.9527,1.2468,36.7141,20985.9923,403.8317,28.3275,3.754,No_Machine_Failure
09-05-2023 19:35:52,Makino-Synth-00273,Shopfloor-L2,135.3674,4.9236,6.3376,13.2566,48.1515,33.4285,1.4427,38.0111,17022.7449,351.5436,39.6228,3.6301,No_Machine_Failure
28-04-2023 22:02:52,Makino-Synth-00995,Shopfloor-L1,182.3217,4.8465,6.1841,23.8599,55.588,40.7393,1.6415,47.3887,15220.2683,287.7356,29.1318,2.8671,No_Machine_Failure
03-12-2022 21:29:55,Makino-Synth-00526,Shopfloor-L4,125.3049,5.6502,5.8703,15.5647,53.0704,38.6793,1.3259,24.4398,22184.2966,345.3991,32.3969,3.2774,No_Machine_Failure
07-09-2024 02:15:42,Makino-Synth-00128,Shopfloor-L5,187.732,7.2756,6.3316,29.1742,48.1836,41.6808,1.9206,49.475,16525.4057,352.8954,27.334,2.624,No_Machine_Failure
13-08-2021 08:47:17,Makino-Synth-00727,Shopfloor-L5,148.0183,5.6844,6.9665,11.196,52.2616,35.5553,0.7707,29.6017,19254.7196,374.4157,14.0456,3.4159,No_Machine_Failure


In [0]:
from pyspark.sql.functions import col, when, lit

gold_df = gold_df.withColumn(
    "health_score",
    (
        lit(100)
        - when(col("spindle_vibration_m") > 3.0, 30)
            .when(col("spindle_vibration_m") > 2.0, 20)
            .otherwise(0)

        - when(col("tool_vibration_m") > 40, 30)
            .when(col("tool_vibration_m") > 35, 20)
            .otherwise(0)

        - when(col("spindle_bearing_temperature_c") > 45, 30)
            .when(col("spindle_bearing_temperature_c") > 40, 20)
            .otherwise(0)

        - when(col("hydraulic_oil_temperature_c") > 55, 30)
            .when(col("hydraulic_oil_temperature_c") > 50, 20)
            .otherwise(0)

        - when(col("downtime") == "Machine_Failure", 40)
            .otherwise(0)
    )
)

In [0]:
gold_df = gold_df.withColumn(
    "risk_level",
    when(col("health_score") >= 80, "Healthy")
    .when(col("health_score") >= 50, "Warning")
    .otherwise("Critical")
)

In [0]:
display(
    gold_df.select(
        "machine_id",
        "health_score",
        "risk_level",
        "downtime"
    )
)

machine_id,health_score,risk_level,downtime
Makino-Synth-00440,100,Healthy,No_Machine_Failure
Makino-Synth-00145,80,Healthy,No_Machine_Failure
Makino-Synth-00879,30,Critical,No_Machine_Failure
Makino-Synth-00684,80,Healthy,No_Machine_Failure
Makino-Synth-00277,80,Healthy,No_Machine_Failure
Makino-Synth-00008,80,Healthy,No_Machine_Failure
Makino-Synth-00052,80,Healthy,No_Machine_Failure
Makino-Synth-00786,100,Healthy,No_Machine_Failure
Makino-Synth-00291,80,Healthy,No_Machine_Failure
Makino-Synth-00312,80,Healthy,No_Machine_Failure


In [0]:
from pyspark.sql.functions import when, col

gold_df = gold_df.withColumn(
    "maintenance_priority",
    when(col("risk_level") == "Critical", "Immediate")
    .when(col("risk_level") == "Warning", "Schedule Soon")
    .otherwise("Routine")
)

In [0]:
from delta.tables import DeltaTable

gold_path = "abfss://processed-data@adarshpmstorage01.dfs.core.windows.net/gold"

gold_table = DeltaTable.forPath(spark, gold_path)

gold_table.alias("target").merge(
    gold_df.alias("source"),
    "target.machine_id = source.machine_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

In [0]:
gold_parquet_path = "abfss://processed-data@adarshpmstorage01.dfs.core.windows.net/gold_parquet"

gold_df.write.mode("overwrite").parquet(gold_parquet_path)